# Single-command GitHub wheel GPU acceptance
No loader, no Kaggle Inputs. CUDA/NCCL inside the wheel; existing Torch dependencies must remain unchanged.


In [ ]:
import importlib.metadata, json, subprocess, sys, time
from pathlib import Path
names=("torch", "nvidia-nccl-cu12", "nvidia-cuda-runtime-cu12")
before={name:importlib.metadata.version(name) for name in names}
started=time.monotonic()
subprocess.run([sys.executable,"-m","pip","install",'multigpubfs[cayleypy] @ https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261013-v2/multigpubfs-0.2.0.dev2-py3-none-linux_x86_64.whl#sha256=ddcbca720876f87fc048c107fb8227d42bc3f6e3ebb9894e87d34db0e405ffac'],check=True,timeout=600)
after={name:importlib.metadata.version(name) for name in names}
assert before==after,(before,after)
import multigpubfs, torch
assert importlib.metadata.version("multigpubfs")=="0.2.0.dev2"
assert torch.cuda.device_count()==2
from multigpubfs.native_distribution import native_runtime
native,env=native_runtime()
manifest=json.loads((Path(native).parents[1]/"manifest.json").read_text())
assert manifest["bundled_runtime"]
assert Path(native).stat().st_mode & 0o111
assert str(Path(native).parents[1]/"lib")==env["LD_LIBRARY_PATH"].split(":")[0]
torch.cuda.synchronize()
dependency_receipt=dict(status="VERIFIED_KAGGLE_SINGLE_PIP_INSTALL",before=before,after=after,module=multigpubfs.__file__,manifest=manifest,install_seconds=time.monotonic()-started)
Path("/kaggle/working/pip-install-acceptance.json").write_text(json.dumps(dependency_receipt,indent=2))
print(dependency_receipt["status"],flush=True)


In [ ]:
print("IMPORT_START", flush=True)
import multigpubfs, cayleypy
multigpubfs.enable_native()
print("NATIVE_READY", flush=True)

# Change these settings, then Run All. Start with the small default grid.
FAMILY = "lrx"  # "lrx" or "rapaport_m2"
N_MIN, N_MAX = 2, 5
PAIR_SECONDS = 120
TOTAL_SECONDS = 600


In [ ]:
for FAMILY in ("lrx", "rapaport_m2"):
    import csv, hashlib, json, time
    from pathlib import Path
    import torch
    from multigpubfs.generic_session import persistent_graph_workers
    
    root = Path("/kaggle/working/pair-sweep") / FAMILY
    root.mkdir(parents=True, exist_ok=True)
    configuration = dict(family=FAMILY, n_min=N_MIN, n_max=N_MAX,
                         pair_seconds=PAIR_SECONDS, total_seconds=TOTAL_SECONDS)
    assert FAMILY in ("lrx", "rapaport_m2")
    assert 2 <= N_MIN <= N_MAX and PAIR_SECONDS > 0 and TOTAL_SECONDS > 0
    assert torch.cuda.device_count() > 0, "Enable GPU in Kaggle Settings"
    constructor = getattr(cayleypy.PermutationGroups, FAMILY)
    config_path = root / "configuration.json"
    if config_path.exists():
        assert json.loads(config_path.read_text()) == configuration, "Use a new output directory for changed settings"
    else:
        config_path.write_text(json.dumps(configuration, indent=2))
    rows, layers = [], []
    started = time.monotonic()
    
    # One resident rank pool for the sweep; admission still checks actual resources.
    with persistent_graph_workers():
        for n in range(N_MIN, N_MAX + 1):
            for r in range(1, n + 1):
                remaining = TOTAL_SECONDS - (time.monotonic() - started)
                case = root / f"n{n}-r{r}"
                report_path = case / "report.json"
                if report_path.exists():
                    report = json.loads(report_path.read_text())
                    assert report["status"] in ("COMPLETE", "INCOMPLETE")
                    wall = None  # Reused receipt: do not invent elapsed time.
                elif remaining < 10:
                    rows.append(dict(n=n, r=r, status="PENDING", eccentricity=None,
                                     completed_depth=None, states=None, api_seconds=None,
                                     bfs_seconds=None, tuning_seconds=None))
                    continue
                else:
                    # CayleyPy lrx() asserts n>=3; extend the same L/R/X definition at n=2.
                    definition = (cayleypy.CayleyGraphDef.create(
                        [[1, 0], [1, 0], [1, 0]], central_state=[0, 1],
                        generator_names=["L", "R", "X"], name="lrx-2")
                        if FAMILY == "lrx" and n == 2 else constructor(n))
                    graph = cayleypy.CayleyGraph(definition, device="cpu")
                    start = torch.tensor(list(range(n-r+1)) + [n-r]*(r-1), dtype=torch.int64)
                    print("PAIR_START", n, r, flush=True)
                    at = time.monotonic()
                    result = graph.bfs(start_states=start, output=case,
                        native_options=multigpubfs.NativeOptions(
                            max_seconds=max(1, int(min(PAIR_SECONDS, remaining))), autotune=True))
                    wall = time.monotonic() - at
                    report = result.report
                sizes = report["layer_sizes"]
                depth = len(sizes) - 1
                rows.append(dict(n=n, r=r, status=report["status"],
                    eccentricity=depth if report["status"] == "COMPLETE" else None,
                    completed_depth=depth, states=sum(sizes), api_seconds=wall,
                    bfs_seconds=report.get("bfs_seconds"),
                    tuning_seconds=report.get("autotune", {}).get("seconds")))
                times = report.get("layer_seconds", [])
                cumulative = 0
                for d, size in enumerate(sizes):
                    cumulative += size
                    layers.append(dict(n=n, r=r, depth=d, states=size,
                        cumulative_states=cumulative,
                        expansion_seconds=times[d-1] if d > 0 and d-1 < len(times) else None))
                print(n, r, report["status"], depth, sum(sizes), flush=True)
    
    for name, records in (("results.csv", rows), ("layers.csv", layers)):
        with (root/name).open("w", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(records[0]) if records else [])
            writer.writeheader(); writer.writerows(records)
    manifest = {str(f.relative_to(root)): hashlib.sha256(f.read_bytes()).hexdigest()
                for f in sorted(root.rglob("*")) if f.is_file() and f.name != "checksums.json"}
    (root/"checksums.json").write_text(json.dumps(manifest, indent=2))
    import pandas as pd
    display(pd.DataFrame(rows))
    print("Saved:", root, "GPU count:", torch.cuda.device_count())
    
    from multigpubfs.graph_definition import from_cayleypy
    for record in rows:
        assert record["status"] == "COMPLETE", record
        n, r = record["n"], record["r"]
        definition = (cayleypy.CayleyGraphDef.create([[1,0]]*3, central_state=[0,1],
                      generator_names=["L","R","X"], name="lrx-2")
                      if FAMILY == "lrx" and n == 2 else constructor(n))
        g = cayleypy.CayleyGraph(definition, device="cpu")
        action = json.loads(from_cayleypy(g).to_json())["action"]
        front = [tuple(list(range(n-r+1))+[n-r]*(r-1))]; seen=set(front); expected=[]
        while front:
            expected.append(len(front)); following=[]
            for state in front:
                for permutation in action["generators"]:
                    child=tuple(state[i] for i in permutation)
                    if child not in seen:seen.add(child);following.append(child)
            front=following
        report=json.loads((root/f"n{n}-r{r}"/"report.json").read_text())
        assert report["layer_sizes"] == expected, (FAMILY,n,r,report["layer_sizes"],expected)
    assert len(rows) == 14
    (root/"oracle-acceptance.json").write_text(json.dumps(dict(status="VERIFIED",family=FAMILY,pairs=len(rows),gpu_count=torch.cuda.device_count())))
    print("ALL_PAIRS_ORACLE_VERIFIED", FAMILY, flush=True)


In [ ]:
import os, subprocess, sys
from pathlib import Path
venv=Path("/tmp/multigpubfs-clean-shell")
clean=dict(os.environ)
for name in ("PYTHONPATH","PYTHONHOME"):
    clean.pop(name,None)
subprocess.run([sys.executable,"-m","venv","--without-pip",str(venv)],check=True,timeout=120,env=clean)
python=str(venv/"bin/python")
subprocess.run([sys.executable,"-m","pip","--python",python,"install",'multigpubfs @ https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261013-v2/multigpubfs-0.2.0.dev2-py3-none-linux_x86_64.whl#sha256=ddcbca720876f87fc048c107fb8227d42bc3f6e3ebb9894e87d34db0e405ffac'],check=True,timeout=600)
script=Path("/tmp/multigpubfs-clean-shell-gate.py")
script.write_text("import json, os, subprocess, sys\nfrom pathlib import Path\nimport multigpubfs\nfrom multigpubfs import GraphDefinition,run_graph\nfrom multigpubfs.generic_session import persistent_graph_workers\nfrom multigpubfs.native_distribution import native_runtime\nnative,env=native_runtime()\nassert not any(os.environ.get(k) for k in ('PYTHONPATH','MGBFS_EXECUTABLE','MGBFS_CUDA_LIB_DIR','MGBFS_CUDART_LIB_DIR'))\nassert os.access(native,os.X_OK)\nassert 'site-packages' in multigpubfs.__file__\ng=GraphDefinition.permutation([[1,2,3,4,0],[4,0,1,2,3],[1,0,2,3,4]],[0,1,2,3,4],name='clean-shell-lrx5',generator_names=['L','R','X'],expected_max_unique_states=120)\noracle=g.exact_layers(120)\nroot=Path('/kaggle/working/clean-linux-shell');root.mkdir()\ncases=[]\nwith persistent_graph_workers():\n for devices in ([0],[0,1]):\n  out=root/('gpu'+str(len(devices)))\n  report=run_graph(g,out,devices=devices,max_seconds=60)\n  assert report['status']=='COMPLETE' and report['layer_sizes']==list(map(len,oracle)),report\n  states=json.loads((out/'states.json').read_text())\n  assert sorted(states['current'])==sorted(oracle[-1])\n  cases.append(dict(devices=devices,layer_sizes=report['layer_sizes'],runtime_identity=report['runtime_identity']))\nmatrix=GraphDefinition.matrix(2,2,[([1,1,0,1],3),([1,2,0,1],3),([1,0,1,1],3),([1,0,2,1],3)],[1,0,0,1],name='clean-shell-sl2-3',expected_max_unique_states=24)\nmatrix_oracle=matrix.exact_layers(24)\nwith persistent_graph_workers():\n for devices in ([0],[0,1]):\n  out=root/('matrix'+str(len(devices)))\n  report=run_graph(matrix,out,devices=devices,max_seconds=60)\n  assert report['status']=='COMPLETE' and report['layer_sizes']==list(map(len,matrix_oracle)),report\n  states=json.loads((out/'states.json').read_text())\n  assert sorted(states['current'])==sorted(matrix_oracle[-1])\n  cases.append(dict(family='matrix',devices=devices,layer_sizes=report['layer_sizes'],runtime_identity=report['runtime_identity']))\n out=root/'resource'\n report=run_graph(g,out,devices=[0,1],capacity=3,shards=4,backend='generic',_batch=1,max_seconds=60)\n assert report['status']=='INCOMPLETE' and report['reason'].startswith('RESOURCE_'),report\n depth=len(report['layer_sizes'])-1\n assert report['layer_sizes']==list(map(len,oracle[:depth+1]))\n states=json.loads((out/'states.json').read_text())\n assert sorted(states['current'])==sorted(oracle[depth])\n cases.append(dict(family='resource',devices=[0,1],layer_sizes=report['layer_sizes'],status=report['status'],reason=report['reason']))\ndefinition=root/'cli-definition.json';definition.write_text(g.to_json())\ncli=Path(sys.executable).parent/'multigpubfs'\nprocess=subprocess.run([str(cli),str(definition),str(root/'cli'),'--devices','0,1','--seconds','60'],text=True,capture_output=True,check=True)\nreport=json.loads(process.stdout);assert report['status']=='COMPLETE' and report['layer_sizes']==list(map(len,oracle))\nreceipt=dict(status='VERIFIED_FRESH_LINUX_SHELL_ONE_TWO_GPU',python=sys.executable,module=multigpubfs.__file__,cases=cases,cli_layer_sizes=report['layer_sizes'])\n(root/'shell-acceptance.json').write_text(json.dumps(receipt,indent=2))\nprint(receipt['status'])\n")
clean=dict(os.environ)
for name in ("PYTHONPATH","MGBFS_EXECUTABLE","MGBFS_CUDA_LIB_DIR","MGBFS_CUDART_LIB_DIR"):
    clean.pop(name,None)
subprocess.run([python,str(script)],check=True,timeout=240,cwd="/tmp",env=clean)
subprocess.run([sys.executable,"-m","pip","--python",python,"check"],check=True,timeout=60,env=clean)
